# Fine-tune BioLinkBERT for Clinical Trial Retrieval

This notebook fine-tunes `michiyasunaga/BioLinkBERT-base` as a bi-encoder using
`MultipleNegativesRankingLoss` with hard negatives.

**Setup:**
1. Upload `train_pairs.jsonl` and `val_pairs.jsonl` to Google Drive
2. Runtime > Change runtime type > **A100 GPU**
3. Run all cells

**What you get back:**
- `fine-tuned-biolinkbert.zip` — download to local `models/embeddings/fine-tuned/`

## Step 1: Verify GPU and install packages

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU! Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
!pip install -q sentence-transformers datasets pyyaml mlflow

## Step 2: Load data from Google Drive

1. Upload `train_pairs.jsonl` and `val_pairs.jsonl` to your Google Drive root (or a subfolder)
2. Update `DRIVE_FOLDER` below if you used a subfolder

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

DRIVE_FOLDER = "/content/drive/MyDrive"  # update if files are in a subfolder

os.makedirs("data/training", exist_ok=True)
!cp "{DRIVE_FOLDER}/train_pairs.jsonl" data/training/
!cp "{DRIVE_FOLDER}/val_pairs.jsonl" data/training/

# Verify
for f in ["data/training/train_pairs.jsonl", "data/training/val_pairs.jsonl"]:
    size = os.path.getsize(f) / 1e6
    print(f"  {f}: {size:.1f} MB")

## Step 3: Load data and model

In [ ]:
import json
import random
import logging

from datasets import load_dataset
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

In [ ]:
# Config — matches configs/training/embeddings.yaml
CONFIG = {
    "model_name": "michiyasunaga/BioLinkBERT-base",
    "max_seq_length": 512,
    "epochs": 3,
    "batch_size": 32,
    "learning_rate": 2e-5,
    "warmup_ratio": 0.1,
    "weight_decay": 0.01,
    "fp16": True,
    "loss_scale": 20.0,
    "logging_steps": 50,
    "eval_steps": 1000,
    "save_steps": 1000,
    "save_total_limit": 3,
    "max_val_samples": 5000,
    "output_dir": "models/fine-tuned",
}

In [ ]:
# Load model
model = SentenceTransformer(CONFIG["model_name"], device="cuda")
model.max_seq_length = CONFIG["max_seq_length"]
print(f"Model: {CONFIG['model_name']}")
print(f"Dimension: {model.get_sentence_embedding_dimension()}, max_seq_length: {model.max_seq_length}")

In [ ]:
# Load training data
dataset = load_dataset("json", data_files={
    "train": "data/training/train_pairs.jsonl",
    "val": "data/training/val_pairs.jsonl",
})

train_ds = dataset["train"]
val_ds = dataset["val"]
print(f"Raw: {len(train_ds):,} train, {len(val_ds):,} val")

# Filter empty negatives
train_ds = train_ds.filter(lambda x: x["negative"] and len(x["negative"].strip()) > 0)
val_ds = val_ds.filter(lambda x: x["negative"] and len(x["negative"].strip()) > 0)
print(f"After filtering: {len(train_ds):,} train, {len(val_ds):,} val")

# Rename for sentence-transformers
train_ds = train_ds.rename_column("query", "anchor")
val_ds = val_ds.rename_column("query", "anchor")
train_ds = train_ds.select_columns(["anchor", "positive", "negative"])
val_ds = val_ds.select_columns(["anchor", "positive", "negative"])

In [ ]:
# Sanity check: print 5 examples
for i in range(5):
    row = train_ds[i]
    print(f"\n--- Example {i+1} ---")
    print(f"  Anchor:   {row['anchor'][:100]}")
    print(f"  Positive: {row['positive'][:100]}...")
    print(f"  Negative: {row['negative'][:100]}...")

## Step 4: Build evaluator

In [ ]:
# Build IR evaluator from validation data
val_rows = []
with open("data/training/val_pairs.jsonl") as f:
    for line in f:
        try:
            val_rows.append(json.loads(line))
        except json.JSONDecodeError:
            continue

val_rows = [r for r in val_rows if r.get("negative", "").strip()]

rng = random.Random(42)
if len(val_rows) > CONFIG["max_val_samples"]:
    val_rows = rng.sample(val_rows, CONFIG["max_val_samples"])

queries, corpus, relevant_docs = {}, {}, {}
for i, row in enumerate(val_rows):
    qid = f"q{i}"
    cid = f"{row.get('nct_id', 'unknown')}_{i}"
    queries[qid] = row["query"]
    corpus[cid] = row["positive"]
    relevant_docs[qid] = {cid}

evaluator = InformationRetrievalEvaluator(
    queries=queries, corpus=corpus, relevant_docs=relevant_docs,
    mrr_at_k=[10], ndcg_at_k=[10],
    accuracy_at_k=[1, 3, 5, 10],
    precision_recall_at_k=[1, 3, 5, 10],
    name="val-retrieval", batch_size=64,
)
print(f"IR evaluator: {len(queries)} queries, {len(corpus)} corpus docs")

# Loss
loss = MultipleNegativesRankingLoss(model=model, scale=CONFIG["loss_scale"])

# Training args
training_args = SentenceTransformerTrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    learning_rate=CONFIG["learning_rate"],
    warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=CONFIG["weight_decay"],
    fp16=CONFIG["fp16"],
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=CONFIG["save_total_limit"],
    logging_steps=CONFIG["logging_steps"],
    load_best_model_at_end=True,
    metric_for_best_model="val-retrieval_cosine_ndcg@10",
    report_to="none",
)

# Print training plan
steps_per_epoch = len(train_ds) // CONFIG["batch_size"]
total_steps = steps_per_epoch * CONFIG["epochs"]
gpu_name = torch.cuda.get_device_name(0)
est_speed = 0.02 if "A100" in gpu_name else 0.15  # seconds per step
print(f"GPU: {gpu_name}")
print(f"Steps/epoch: {steps_per_epoch:,}")
print(f"Total steps: {total_steps:,}")
print(f"Estimated time: ~{total_steps * est_speed / 3600:.1f} hours")

In [ ]:
# Loss
loss = MultipleNegativesRankingLoss(model=model, scale=CONFIG["loss_scale"])

# Training args
training_args = SentenceTransformerTrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    learning_rate=CONFIG["learning_rate"],
    warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=CONFIG["weight_decay"],
    fp16=CONFIG["fp16"],
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=CONFIG["save_total_limit"],
    logging_steps=CONFIG["logging_steps"],
    load_best_model_at_end=True,
    metric_for_best_model="val-retrieval_cosine_ndcg@10",
    report_to="none",  # no MLflow on Colab
)

# Print training plan
steps_per_epoch = len(train_ds) // CONFIG["batch_size"]
total_steps = steps_per_epoch * CONFIG["epochs"]
print(f"Steps/epoch: {steps_per_epoch:,}")
print(f"Total steps: {total_steps:,}")
print(f"Estimated time (T4): ~{total_steps * 0.15 / 3600:.1f} hours")

In [ ]:
# Create trainer and start training
import time

trainer = SentenceTransformerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    loss=loss,
    evaluator=evaluator,
)

start = time.time()
trainer.train()
elapsed = time.time() - start
print(f"\nTraining completed in {elapsed / 60:.1f} minutes")

## Step 6: Evaluate and save

In [ ]:
# Final evaluation
eval_results = evaluator(model)

print("\n" + "=" * 60)
print("FINAL EVALUATION RESULTS")
print("=" * 60)
for key in sorted(eval_results):
    if "cosine" in key:
        print(f"  {key}: {eval_results[key]:.4f}")

# Key metrics
ndcg = eval_results.get("val-retrieval_cosine_ndcg@10", "N/A")
mrr = eval_results.get("val-retrieval_cosine_mrr@10", "N/A")
recall_1 = eval_results.get("val-retrieval_cosine_recall@1", "N/A")
recall_10 = eval_results.get("val-retrieval_cosine_recall@10", "N/A")

print(f"\nSummary:")
print(f"  NDCG@10:  {ndcg}")
print(f"  MRR@10:   {mrr}")
print(f"  Recall@1: {recall_1}")
print(f"  Recall@10:{recall_10}")

In [ ]:
# Save model and metadata
from datetime import datetime, timezone

model.save(CONFIG["output_dir"])
print(f"Model saved to {CONFIG['output_dir']}/")

# Save metadata
metadata = {
    "training_date": datetime.now(timezone.utc).isoformat(),
    "base_model": CONFIG["model_name"],
    "dataset_size": {"train": len(train_ds), "val": len(val_ds)},
    "hyperparameters": {
        "epochs": CONFIG["epochs"],
        "batch_size": CONFIG["batch_size"],
        "learning_rate": CONFIG["learning_rate"],
        "warmup_ratio": CONFIG["warmup_ratio"],
        "weight_decay": CONFIG["weight_decay"],
        "loss": "MultipleNegativesRankingLoss",
        "loss_scale": CONFIG["loss_scale"],
        "max_seq_length": CONFIG["max_seq_length"],
        "fp16": CONFIG["fp16"],
    },
    "device": "cuda",
    "gpu": torch.cuda.get_device_name(0),
    "training_time_minutes": round(elapsed / 60, 1),
    "eval_metrics": {k: v for k, v in eval_results.items() if "cosine" in k},
}

with open(f"{CONFIG['output_dir']}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
print("Metadata saved.")

## Step 7: Download the fine-tuned model

Saves the model to Google Drive as a zip. On your local machine:
1. Download `fine-tuned-biolinkbert.zip` from Drive
2. Unzip to `models/embeddings/fine-tuned/`
3. Rebuild FAISS index: `python scripts/build_index.py --skip-bm25`
4. Re-run comparison: `python scripts/compare_methods.py`

import shutil

# Zip the model directory
zip_path = shutil.make_archive("fine-tuned-biolinkbert", "zip", CONFIG["output_dir"])
zip_size = os.path.getsize(zip_path) / 1e6
print(f"Zipped model: {zip_path} ({zip_size:.1f} MB)")

# Save to Google Drive
drive_dest = "/content/drive/MyDrive/fine-tuned-biolinkbert.zip"
shutil.copy(zip_path, drive_dest)
print(f"Saved to Google Drive: {drive_dest}")
print("\nDownload this file from Drive, then unzip to models/embeddings/fine-tuned/")

In [ ]:
# Alternative: save to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r {CONFIG['output_dir']}/* "/content/drive/MyDrive/TrialMine/fine-tuned-model/"
# print("Model copied to Google Drive")